# Personalized Product Recommendation System

**Portfolio case study: collaborative filtering and matrix factorization for e-commerce personalization**

This notebook is a cleaned, recruiter-facing version of an elective recommendation-systems project. It uses Amazon Electronics ratings to compare popularity, neighborhood-based collaborative filtering, and SVD matrix factorization. The emphasis is not just on model scores, but on what the results imply for a deployable recommendation strategy.

## 1. Business question

How should an e-commerce platform personalize product recommendations when its user-item interaction matrix is extremely sparse?

We evaluate a non-personalized popularity baseline and three personalized approaches, then translate the offline results into a practical routing strategy for cold-start and returning users.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = Path('../data/ratings_Electronics.csv')
COLUMNS = ['user_id', 'prod_id', 'rating', 'timestamp']

## 2. Data and filtering

The original source contains **7,824,482 ratings**, **4,201,696 users**, and **476,002 products**. The project filters to users with at least 50 ratings and products with at least 5 ratings, leaving **65,290 interactions** across **1,540 users** and **5,689 products**.

This makes neighborhood modeling tractable, but it creates **selection bias toward active users and popular products**. That limitation is carried through the interpretation rather than treated as a harmless preprocessing detail.

In [ ]:
df = pd.read_csv(DATA_PATH, names=COLUMNS).drop(columns='timestamp')
user_counts = df['user_id'].value_counts()
df = df[df['user_id'].isin(user_counts[user_counts >= 50].index)]
product_counts = df['prod_id'].value_counts()
df_final = df[df['prod_id'].isin(product_counts[product_counts >= 5].index)].copy()

print(df_final.shape)
print(df_final[['user_id','prod_id']].nunique())
print(df_final['rating'].describe())

### Sparsity

The filtered user-item matrix has density **0.7452%** and sparsity **99.2548%**. That explains why neighborhood methods can still encounter insufficient overlap even after aggressive filtering.

In [ ]:
n_users = df_final['user_id'].nunique()
n_items = df_final['prod_id'].nunique()
density = len(df_final) / (n_users * n_items)
print(f"Matrix density: {density:.4%}")
print(f"Matrix sparsity: {1-density:.4%}")

## 3. Popularity baseline

A popularity recommender is not personalized, but it is a useful **cold-start fallback** when a user has little or no interaction history. We require a minimum interaction count so a product cannot rank highly on the strength of only a handful of ratings.

In [ ]:
def top_n_products(data, n=5, min_interactions=50):
    summary = data.groupby('prod_id')['rating'].agg(['mean','count'])
    eligible = summary[summary['count'] >= min_interactions]
    return eligible.sort_values(['mean','count'], ascending=False).head(n)

top_n_products(df_final, 5, 50)

## 4. Personalized models

The original project uses `scikit-surprise` with an 80/20 random interaction split. Relevant items are defined as actual ratings ≥ 3.5; Precision@10, Recall@10, F1@10, and RMSE are reported.

> **Portfolio audit correction:** the original user-user grid search selected `msd`, but the subsequent “optimized” model was instantiated with `cosine`. The corrected code below uses the grid-search winner. The historical result table is kept separately and labels the original output as a tuned variant rather than falsely claiming it was the exact grid-search configuration.

In [ ]:
# Requires scikit-surprise
from surprise import Dataset, Reader, KNNBasic, SVD, accuracy
from surprise.model_selection import train_test_split, GridSearchCV

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df_final[['user_id','prod_id','rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
# Corrected optimized user-user configuration
user_grid = {
    'k': [20, 30, 40],
    'min_k': [3, 6, 9],
    'sim_options': {'name': ['msd', 'cosine'], 'user_based': [True]},
}
user_search = GridSearchCV(KNNBasic, user_grid, measures=['rmse'], cv=3, n_jobs=-1)
user_search.fit(data)
print(user_search.best_params['rmse'])

best_user = KNNBasic(**user_search.best_params['rmse'], verbose=False)
best_user.fit(trainset)

## 5. Historical model comparison

The table below reproduces the executed results from the submitted notebook. These are retained as historical outputs; the user-user tuned row is explicitly qualified because of the configuration mismatch identified above.

In [ ]:
results = pd.DataFrame([
    ['User-user KNN — baseline', 1.0012, .855, .858, .856],
    ['User-user KNN — tuned variant*', .9526, .847, .894, .870],
    ['Item-item KNN — baseline', .9950, .838, .845, .841],
    ['Item-item KNN — optimized', .9576, .839, .880, .859],
    ['SVD — baseline', .8882, .853, .880, .866],
    ['SVD — optimized', .8822, .854, .884, .869],
], columns=['Model','RMSE','Precision@10','Recall@10','F1@10'])
results

### Model selection

**Optimized SVD is the strongest overall model in the submitted results.** It has the lowest RMSE (0.8822), with Precision@10 and Recall@10 that remain competitive with the best KNN scores. More importantly, the original KNN examples sometimes returned `Not enough neighbors`, while SVD can estimate preferences through learned latent factors without requiring a sufficiently large local neighborhood for every prediction.

The highest Recall@10 belongs to the tuned user-user variant (0.894), but its lower precision and neighbor-availability problem make it less attractive as the sole production model.

## 6. Business recommendations

**Route recommendations by user state.** Use popularity for new users, matrix factorization for users with enough historical signal, and item-item similarity for “similar product” modules or explainable recommendations. If a neighborhood model cannot find enough support, automatically fall back instead of exposing a low-confidence score.

**Measure online business impact.** RMSE and ranking metrics are useful offline diagnostics, but a deployment decision should be validated with A/B tests measuring click-through rate, add-to-cart rate, conversion, revenue per session, and repeat engagement. Recommendation coverage should also be monitored to prevent excessive concentration on a small set of popular products.

**Evaluate sparse-history users separately.** Because the modeling subset excludes users with fewer than 50 ratings, aggregate metrics do not tell us how the system performs on the customers who most need cold-start handling. Segment evaluation by interaction-history bucket before launch.

## 7. Limitations and next steps

- The source contains IDs and explicit ratings only; no product metadata, text, images, price, availability, or session context is available.
- Filtering improves tractability but biases the sample toward active users and popular products.
- A random interaction split does not model temporal deployment as well as a time-aware holdout.
- The course evaluation ranks only held-out rated items per user rather than a large candidate set of truly unseen products. A stronger next iteration would add NDCG@K, Hit Rate@K, catalog coverage, and history-segment analysis.
- For a production-scale system, candidate generation and ranking would typically be separated rather than scoring every unseen catalog item in a notebook loop.